<a href="https://colab.research.google.com/github/narpavi-ai/cctp-481-notes/blob/main/notebooks/02-first-agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 2 — Build Your First Agent

**CCTP 481: Building Your First AI Agent · Module 2**

---

### Where we left off

You own a food truck in Edmonton's river valley. In Lab 1 you asked a plain
model your three morning questions and it struck out on all three:

| | You asked | It said |
|---|---|---|
| 1 | What's the weather doing? | *"I don't have access to real-time weather data."* |
| 2 | What have I got left? | *"I don't have access to your inventory."* |
| 3 | Should I open at Hawrelak today? | …it couldn't, because that one needs both. |

Two gaps: **it can't see the world**, and **it can't see your business**.

### What you'll do here

Close both. You'll give the model two tools — a **live weather lookup** that
calls a real weather service over the internet, and a **stock lookup** that reads
your inventory — and then watch it decide *on its own* which one to call.

That decision is the whole difference between a model and an agent.

By the end you will have:

1. A tool that fetches real Edmonton weather, right now, for free
2. An agent that picks its own tools
3. The **ReAct loop** from CCTP 480, traced step by step in real output
4. A safety belt that stops a runaway agent from burning your quota

⏱️ About 35 minutes.

<p align="center">
<img src="https://raw.githubusercontent.com/narpavi-ai/cctp-481-notes/main/images/lab2.png" alt="The same food truck, now wired to a live weather tile and a stock tile - its first two tools" width="640">
</p>

### Setup

In [ ]:
%pip install -q -U "langchain[google-genai]>=1.3,<2"

print("✅ Installed.")

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
    print("✅ Key loaded from Colab Secrets.")
except Exception:
    import getpass
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Paste your Google AI Studio API key: ")
    print("✅ Key loaded for this session only.")

In [ ]:
from IPython.display import Markdown, display

# One look for everything the model and the tools say, so a student can tell at
# a glance where the notebook stops talking and the model starts.

ANSWER_LIMIT = 1500


def _text_of(message):
    """The readable text of a model message, whatever shape it arrives in.

    Gemini 3.x returns .content as a LIST of blocks, not a string, so the
    obvious str(response.content) prints a Python list with a base64 signature
    inside it. Everything below goes through here.

    Deliberately does NOT touch .text: calling it is deprecated in LangChain 1.x
    and printed a warning above every single answer.
    """
    content = getattr(message, "content", message)
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for block in content:
            if isinstance(block, str):
                parts.append(block)
            elif isinstance(block, dict) and block.get("type") == "text":
                parts.append(block.get("text", ""))
        return "\n\n".join(p for p in parts if p)
    return str(content)


def _card(body, label=None, icon="", limit=ANSWER_LIMIT):
    """A labelled, indented block. Markdown inside still renders."""
    body = _text_of(body).strip()
    if not body:
        body = "*(nothing came back)*"
    if len(body) > limit:
        body = body[:limit].rstrip() + f"\n\n*… trimmed here — {len(body):,} characters in full*"
    lines = ["> " + line for line in body.splitlines()]
    if label:
        lines = [f"> {icon} **{label}**".replace(">  ", "> "), ">"] + lines
    return "\n".join(lines)


def show(response, label="Model answer"):
    """Display a model response as a readable answer card."""
    display(Markdown(_card(response, label, icon="🤖")))


def show_text(text, label=None, icon="📄"):
    """Display plain text - a tool result, a lookup - in the same card."""
    display(Markdown(_card(text, label, icon=icon, limit=2500)))


def show_trace(result, label="Agent trace"):
    """Render every step the agent took, in order, as readable cards."""
    msgs = result["messages"]
    out = [f"#### 🔍 {label} — {len(msgs)} steps", ""]

    for i, m in enumerate(msgs, 1):
        kind = type(m).__name__.replace("Message", "").upper()

        if kind == "HUMAN":
            out += [_card(m, f"{i} · You asked", icon="👤", limit=600), ""]

        elif kind == "TOOL":
            name = getattr(m, "name", "tool")
            out += [_card(m, f"{i} · Tool returned — {name}", icon="🛠️", limit=800), ""]

        elif kind == "AI":
            calls = getattr(m, "tool_calls", None)
            if calls:
                steps = []
                for tc in calls:
                    args = ", ".join(f"{k}={v!r}" for k, v in tc["args"].items())
                    steps.append(f"`{tc['name']}({args})`")
                said = _text_of(m).strip()
                body = "\n\n".join(steps + ([said] if said else []))
                out += [_card(body, f"{i} · Model called a tool", icon="🔧", limit=600), ""]
            else:
                out += [_card(m, f"{i} · Model answered", icon="🤖", limit=900), ""]

        else:
            out += [_card(m, f"{i} · {kind}", limit=600), ""]

    display(Markdown("\n".join(out)))


print("✅ Display helpers ready — use show() instead of print() from here on.")


#### Create the model

Every notebook is its own Colab runtime, so `model` does not carry over from the
last lab — you build it again here. Same one line as Lab 1.

`gemini-3.5-flash-lite` is the pin, and the reason is **quota, not cleverness**:
on the free tier the full Flash models allow **20 requests a day** and these labs
need roughly 70. Check your own limits at <https://aistudio.google.com/rate-limit>.


In [ ]:
from langchain.chat_models import init_chat_model

MODEL = "google_genai:gemini-3.5-flash-lite"

model = init_chat_model(MODEL)
print(f"✅ Model ready: {MODEL}")


### Step 1 — Watch it fail one more time

Not to be cruel. You need this answer fresh in your mind, because in about ten
minutes the *same question* to the *same model* is going to work.

In [ ]:
show(model.invoke("What's the current weather in Edmonton?"))

### Step 2 — Your first tool: real weather, no API key

A **tool** is a normal Python function with a `@tool` decorator on top. That's it.

This one calls [Open-Meteo](https://open-meteo.com) — a free weather service that
needs **no API key and no signup** for non-commercial use. You're about to give a
language model the ability to fetch a fact that did not exist when it was trained.

Read the **docstring** carefully. It looks like a comment for humans. It isn't.

In [ ]:
import requests
from langchain.tools import tool

# Open-Meteo returns a numeric WMO code. This turns it into English.
WEATHER_CODES = {
    0: "clear sky", 1: "mainly clear", 2: "partly cloudy", 3: "overcast",
    45: "fog", 48: "freezing fog",
    51: "light drizzle", 53: "drizzle", 55: "heavy drizzle",
    61: "light rain", 63: "rain", 65: "heavy rain",
    66: "freezing rain", 67: "heavy freezing rain",
    71: "light snow", 73: "snow", 75: "heavy snow", 77: "snow grains",
    80: "rain showers", 81: "heavy rain showers", 82: "violent rain showers",
    85: "snow showers", 86: "heavy snow showers",
    95: "thunderstorm", 96: "thunderstorm with hail", 99: "severe thunderstorm with hail",
}


@tool
def get_weather() -> str:
    """Get the CURRENT weather in Edmonton, Alberta.

    Use this for any question about weather, temperature, wind, rain or snow,
    or whether conditions are suitable for opening the food truck.
    Takes no input - it always reports Edmonton.
    """
    now = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params={
            "latitude": 53.5461, "longitude": -113.4938,
            "current": "temperature_2m,apparent_temperature,precipitation,weather_code,wind_speed_10m",
            "timezone": "America/Edmonton",
        },
        timeout=10,
    ).json()["current"]

    return (
        f"Edmonton weather as of {now['time']}: "
        f"{WEATHER_CODES.get(now['weather_code'], 'unknown conditions')}, "
        f"{now['temperature_2m']}°C (feels like {now['apparent_temperature']}°C), "
        f"wind {now['wind_speed_10m']} km/h, "
        f"precipitation {now['precipitation']} mm."
    )


print("✅ Tool defined. Calling it directly, as a plain Python function:")
show_text(get_weather.invoke({}), "Live Edmonton weather, right now")

**That's real weather in Edmonton, right now.** No key, no account, no bill.

Now look at what the *model* will see when it's deciding whether to use this:

In [ ]:
show_text(get_weather.name, "Tool name the model sees")
show_text(get_weather.description, "The description the model reads - this is your docstring")
show_text(get_weather.args or "(none - this tool needs no input)", "Arguments it has to fill in itself")

**💡 The most important idea in this lab.** That description *is* the docstring
you wrote. The model never sees your code — it sees the name, the description,
and the argument names, and from those alone it decides **whether** to call your
tool and **what** to pass it.

> **A tool's docstring is not documentation. It is a prompt.**

If your agent ignores a tool it should have used, or calls it with nonsense, the
docstring is the first place to look — not the model.

### Step 3 — Build the agent

One function. You pass it a model and a list of tools, and you get back something
that can *decide*.

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    tools=[get_weather],
    system_prompt=(
        "You are an assistant for a food truck in Edmonton's river valley. "
        "When asked about weather or conditions, always use your tools rather "
        "than guessing or telling the user to check a website."
    ),
)

print("✅ Agent built")

In [ ]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "What's the current weather in Edmonton?"}]}
)

show(result["messages"][-1], "The same question that failed in Step 1")

**🎯 Checkpoint.** Same model. Same free API key. Same question it refused ten
minutes ago.

The only thing that changed is that it now has a **tool** — and it worked out for
itself that it should call it. Nobody told it "call `get_weather` now." It read
the description and decided.

### Step 4 — Look inside the loop

`result["messages"]` holds **every step the agent took**, not just the answer.
This is the part a chat box never shows you, and it is the single most useful
debugging habit in this course.

In [ ]:
show_trace(result)

**💡 Read that trace carefully — it *is* the ReAct loop from CCTP 480:**

| Trace step | ReAct name | What happened |
|---|---|---|
| `[1] HUMAN` | — | Your question went in |
| `[2] AI` with `🔧 calls` | **Reason** | The model decided a tool was needed, and chose the arguments |
| `[3] TOOL` | **Act** | Your Python function ran and hit the internet |
| `[4] AI` | **Observe → answer** | The model read the result and wrote a reply |

In CCTP 480 you saw that diagram on a slide. This is the same loop, in your own
output, with your own function in the middle of it.

### Step 5 — A second tool, and a second kind of gap

Weather closed gap one: **the world**. Now close gap two: **your business**.

Notice how different this tool is. It touches no network, has no API, and no
model anywhere was ever trained on it — because it's *yours*.

In [ ]:
# Pretend this is your inventory system.
STOCK = {
    "cinnamon buns": 4,
    "saskatoon berry pies": 11,
    "bison chili": 0,
    "cold brew": 26,
}


@tool
def check_stock(item: str) -> str:
    """Look up how many units of a menu item are currently in the truck.

    Use this whenever someone asks about inventory, stock levels, or whether
    something is available to sell. The item name should be lowercase and
    plural, for example "cinnamon buns".
    """
    count = STOCK.get(item.lower().strip())
    if count is None:
        return f"No menu item called {item!r}. The menu is: {list(STOCK)}"
    return f"{item}: {count} in the truck"


agent = create_agent(
    model=model,
    tools=[get_weather, check_stock],
    system_prompt=(
        "You are an assistant for a food truck in Edmonton's river valley. "
        "Use your tools rather than guessing. Never invent inventory numbers."
    ),
)

print("✅ Agent now has two tools:", [t.name for t in [get_weather, check_stock]])

### Step 6 — Watch it choose

You haven't told it which tool to use. Ask it three different things and watch it
pick — including one where it needs **both**.

In [ ]:
r = agent.invoke({"messages": [{"role": "user", "content": "How many cinnamon buns do I have?"}]})
show_trace(r)

In [ ]:
r = agent.invoke({"messages": [{"role": "user", "content": "Do I have any bison chili left?"}]})
show(r["messages"][-1], "Sold out - and it says so without inventing a number")

Now the question you actually care about — the one from Lab 1 that needed both
gaps closed at once. **Watch the trace, not just the answer.**

In [ ]:
r = agent.invoke({"messages": [{"role": "user", "content":
    "Should I open at Hawrelak Park today? Tell me what you're basing that on."}]})

show_trace(r)

In [ ]:
show(r["messages"][-1], "Its recommendation")

### ⚠️ Now read that answer again, sceptically

It called both tools. It got real weather and a real stock count. Then it told
you whether to open.

**On whose authority?**

You never gave it a rule. There is no line anywhere in this notebook that says
what temperature is too cold, or how few cinnamon buns is too few. The model
made a threshold up — fluently, confidently, and in a tone that sounds like it
knows your business.

It might even be *right*. That's what makes it dangerous.

| It has | It does not have |
|---|---|
| ✅ Live weather — a real fact | ❌ Your closure policy |
| ✅ Your stock count — a real fact | ❌ Your break-even numbers |
| ✅ A confident recommendation | ❌ Any basis for it |

> **This is the gap Lab 3 closes.** Facts aren't judgement. In Module 3 you'll
> give the agent *your handbook* — the actual rule about when you don't open —
> and this same question stops being a guess and starts being a quotation.

Getting an agent to *use tools* is easy. Getting it to **only** claim what it can
actually support is the hard part, and it's most of the rest of this course.

### Step 7 — Watch it recover from its own mistake

Ask for something that isn't on the menu at all.

In [ ]:
r = agent.invoke({"messages": [{"role": "user", "content": "Do we have any poutine?"}]})
show_trace(r)

**💡 Look at what happened in the trace.** The model called `check_stock("poutine")`,
your function returned an error string listing the real menu, and the model
**read that error and recovered** — it didn't crash, and it didn't invent a number.

That's why tools should return *helpful* error strings rather than raising.
The error message is another prompt. Write it for the model as well as for you.

### Step 8 — Runaway agents

An agent loops until *it* decides it's done. If it never decides, it keeps calling
tools, and every loop costs tokens and quota. On a paid account, money.

Here's one deliberately pushed into a loop — with a seatbelt on.

In [ ]:
from langgraph.errors import GraphRecursionError

try:
    agent.invoke(
        {"messages": [{"role": "user", "content":
            "Check the weather at every one of my locations, one at a time, over and over, forever."}]},
        config={"recursion_limit": 6},
    )
except GraphRecursionError:
    print("🛑 Stopped by the recursion limit — exactly as intended.")
    print("Without this, it keeps looping, and keeps spending.")

**💡 `recursion_limit` is a budget, not an error.** Every agent you ever deploy
should have one. It's the cheapest safety mechanism in this course and the one
most often left out.

### Your turn

1. **Add a forecast tool.** `get_weather` only reports *right now*, but you plan
   the truck a day ahead. Open-Meteo returns forecasts from the same URL — swap
   `current=` for `daily=temperature_2m_max,temperature_2m_min,weather_code` and
   add `"forecast_days": 2`. Write it as a second `@tool`, add it to the list,
   and ask *"what's it doing tomorrow?"*
2. **Break a docstring on purpose.** Change `check_stock`'s docstring to something
   useless like `"""Does a thing."""`, rebuild the agent, and ask about stock
   again. Watch it stop calling the tool. Then put the docstring back. This is the
   fastest way to internalise that the docstring is a prompt.
3. **Give it a tool for your own work.** One lookup a colleague asks you for
   often. It can return hard-coded text — the point is the description.

In [ ]:
# Your tools here.

# @tool
# def get_forecast(location: str) -> str:
#     """..."""

# agent = create_agent(model=model, tools=[get_weather, check_stock, ...],
#                      system_prompt="...")

## If something breaks

| What you see | What it means | Fix |
|---|---|---|
| `404 NOT_FOUND` / *"no longer available to new users"* | Google retired that model | Check <https://aistudio.google.com> and edit `MODEL` |
| The agent answers without calling the tool | Your docstring didn't convince it | Make the description say **when** to use the tool, not just what it does |
| `ConnectionError` / `Timeout` on `get_weather` | Colab couldn't reach Open-Meteo | Re-run the cell. If it persists, check <https://open-meteo.com> is up |
| `GraphRecursionError` when you didn't expect it | The agent genuinely looped | Raise the limit *only* after reading the trace to see why |
| `KeyError: 'current'` | Open-Meteo returned an error body | `print(r.json())` inside the tool to see what came back |

## What you learned

- A **tool** is a plain Python function; the `@tool` decorator makes it callable by a model
- **The docstring is a prompt** — it's the entire basis on which the model decides
- Tools close two different gaps: **the world** (live data) and **your business** (private data)
- `result["messages"]` is the **ReAct loop**, traceable step by step
- An agent with facts still **invents judgement** — which is Module 3's problem
- Every agent needs a `recursion_limit`

## References

**LangChain docs**

- [Agents](https://docs.langchain.com/oss/python/langchain/agents) — `create_agent` and what it builds
- [Tools](https://docs.langchain.com/oss/python/langchain/tools) — the `@tool` decorator, schemas, error handling
- [Messages](https://docs.langchain.com/oss/python/langchain/messages) — reading a trace

**Beyond LangChain**

- [ReAct: Synergizing Reasoning and Acting in Language Models](https://arxiv.org/abs/2210.03629) — the paper behind the loop
- [Anthropic — Building Effective Agents](https://www.anthropic.com/engineering/building-effective-agents) — when *not* to build an agent
- [Open-Meteo docs](https://open-meteo.com/en/docs) — the free weather API used here

### The no-code version of this lab

**[`n8n/02-first-agent.json`](n8n/02-first-agent.json)** builds this same agent on
a canvas — same tools, same model, no Python. See **[`n8n/SETUP.md`](n8n/SETUP.md)**.

### Next

**Lab 3 — Tools, Data & Memory.** Your agent forgets you between questions, and it
guesses at decisions it has no rule for. Next you give it your handbook and a
memory, and both of those stop being true.